In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd

#  📊 Raw Data Exploration | Meat Consumption Project

**Description:** 
In this notebook, we load and explore the raw datasets related to global meat consumption. The goal is to understand the structure, completeness, and key variables of each dataset. We also identify any necessary preprocessing steps and visualize the available data to guide the next stages of analysis.

--

**Structure:**
- [Loading raw data](#loading-raw-data)
- [FAO Dataset](#fao-dataset)
- [GDP Dataset](#gdp-dataset)
- [Urban Dataset](#urban-dataset)
- [Edudcation Dataset](#education-dataset)
- [Enviroment Dataset](#enviroment-dataset)
- [Production Dataset](#meat-production-dataset)

--

## Loading raw data

In [33]:
# file dictionary
files = {
    'faostat': '../Data/raw/faostat_meat_kg_per_capita.csv',
    'gdp': '../Data/raw/worldbank_gdp_per_capita.csv',
    'urban':'../Data/raw/worldbank_urban_population.csv',
    'education':'../Data/raw/share-of-the-world-population-with-at-least-basic-education.csv',
    'enviroment':'../Data/raw/food_enviroment_impact.csv',
    'production':'../Data/raw/global-meat-production.csv'
}

#l configuration dictionary
config = {
    'faostat': {},
    'gdp': {'header':4},
    'urban':{'header':4},
    'education':{},
    'enviroment':{},
    'production':{}
    }
# frame work dictinary
dfs = {

    key: pd.read_csv(path,**config.get(key, {}))
    for key, path in files.items()
}

## FAO Dataset

**Description.**
This dataset provides the per capita meat consumption (in kg/year) for different countries and types of meat.


In [96]:
# First we simplify the name of the frame we will explore. 
faostat = dfs['faostat']

In [97]:
# File header structure
faostat.head(3)

,Domain Code,Domain,Area Code (M49),Area,Element Code,Element,Item Code (FBS),Item,Year Code,Year,Unit,Value,Flag,Flag Description,Note
0,FBS,Food Balances (2010-),4,Afghanistan,645,Food supply quantity (kg/capita/yr),S2731,Bovine Meat,2010,2010,kg/cap,4.74,E,Estimated value,NaN
1,FBS,Food Balances (2010-),4,Afghanistan,645,Food supply quantity (kg/capita/yr),S2731,Bovine Meat,2011,2011,kg/cap,4.80,E,Estimated value,NaN
2,FBS,Food Balances (2010-),4,Afghanistan,645,Food supply quantity (kg/capita/yr),S2731,Bovine Meat,2012,2012,kg/cap,4.40,E,Estimated value,NaN


In [98]:
# file tail structure
faostat.tail(3)

,Domain Code,Domain,Area Code (M49),Area,Element Code,Element,Item Code (FBS),Item,Year Code,Year,Unit,Value,Flag,Flag Description,Note
11862,FBS,Food Balances (2010-),716,Zimbabwe,645,Food supply quantity (kg/capita/yr),S2735,"Meat, Other",2020,2020,kg/cap,2.35,E,Estimated value,NaN
11863,FBS,Food Balances (2010-),716,Zimbabwe,645,Food supply quantity (kg/capita/yr),S2735,"Meat, Other",2021,2021,kg/cap,2.35,E,Estimated value,NaN
11864,FBS,Food Balances (2010-),716,Zimbabwe,645,Food supply quantity (kg/capita/yr),S2735,"Meat, Other",2022,2022,kg/cap,2.35,E,Estimated value,NaN


In [99]:
faostat.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11865 entries, 0 to 11864
Data columns (total 15 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Domain Code       11865 non-null  object 
 1   Domain            11865 non-null  object 
 2   Area Code (M49)   11865 non-null  int64  
 3   Area              11865 non-null  object 
 4   Element Code      11865 non-null  int64  
 5   Element           11865 non-null  object 
 6   Item Code (FBS)   11865 non-null  object 
 7   Item              11865 non-null  object 
 8   Year Code         11865 non-null  int64  
 9   Year              11865 non-null  int64  
 10  Unit              11865 non-null  object 
 11  Value             11865 non-null  float64
 12  Flag              11865 non-null  object 
 13  Flag Description  11865 non-null  object 
 14  Note              0 non-null      float64
dtypes: float64(2), int64(4), object(9)
memory usage: 1.4+ MB


In [100]:
faostat.isnull().sum()

Domain Code             0
Domain                  0
Area Code (M49)         0
Area                    0
Element Code            0
Element                 0
Item Code (FBS)         0
Item                    0
Year Code               0
Year                    0
Unit                    0
Value                   0
Flag                    0
Flag Description        0
Note                11865
dtype: int64

In [101]:
print('(rows, columns):',faostat.shape)
print('columns:', faostat.columns)

(rows, columns): (11865, 15)
columns: Index(['Domain Code', 'Domain', 'Area Code (M49)', 'Area', 'Element Code',
       'Element', 'Item Code (FBS)', 'Item', 'Year Code', 'Year', 'Unit',
       'Value', 'Flag', 'Flag Description', 'Note'],
      dtype='object')


In [95]:
# General exploration, number of data and features
print('Unic values in  columns')
print(faostat['Domain'].unique())
print(faostat['Item Code (FBS)'].unique())
print('Countries:', faostat['Area'].nunique())
print('Year range:',faostat['Year'].min(), '-' , faostat['Year'].max())
print('Suspicious values',faostat['Flag'].value_counts())

Unic values in  columns
['Food Balances (2010-)']
['S2731' 'S2732' 'S2733' 'S2734' 'S2735']
Countries: 190
Year range: 2010 - 2022
Suspicious values Flag
E    11865
Name: count, dtype: int64


### Summary of FAOSTAT Dataset
**Dataset size**
- Rows: 11865
- columns: 15

**Key feature/columns**
- `Area`: Country
- `Item`: Meet type
- `Year`: Measure  year
- `Value`: sonsumption (kg/capita/year)

**Periodcoverage**
- Years: 2010 - 2022

**Geography coverage**
- number of countries: 190

**Missing names**
- `Notes` is the only column with `NaN` (irrelevant)

**Relevant notes**
- Some rows contain `Item = Meat, Other`: THis could be excluded for our analysis
- `Element` = "Fuud Supply quality (kg/capita/yr)" is the relevant input for training.

**Suggested actions for processing**
- Filter by `Element`
- Exclude `Item = "Meat, Other"`
- Rename key columns

## GDP Dataset

In [51]:
# First we simplify the name of the frame we will explore. 
gdp = dfs['gdp']

In [54]:
# General exploration, number of data and features
print('(rows, columns):',gdp.shape)
print('The file contains:')
print('columns:', gdp.columns)

(rows, columns): (264, 70)
The file contains:
columns: Index(['Africa Eastern and Southern', 'AFE', 'GDP per capita (current US$)',
       'NY.GDP.PCAP.CD', '186.121834666874', '186.941781389892',
       '197.402402426459', '225.440493639993', '208.999748076',
       '226.876512633529', '240.955232436321', '243.817323446096',
       '257.190081597614', '281.629276716978', '276.782059767758',
       '294.866149448689', '311.51901877983', '389.796972143257',
       '463.549790110905', '479.162164741466', '468.856330923431',
       '518.450612880301', '571.720267490217', '634.561894108334',
       '773.439454072262', '777.833110434383', '725.728116212689',
       '732.588723549442', '650.563519617381', '554.439146516658',
       '578.603956518682', '665.119756482631', '704.466221711',
       '728.549336565123', '822.79386754523', '864.563865250914',
       '733.243855634055', '709.659254937363', '701.041551520198',
       '766.820590836759', '747.069673916573', '767.684192170293',
       

In [55]:
# File header structure
gdp.head(3)

,Africa Eastern and Southern,AFE,GDP per capita (current US$),NY.GDP.PCAP.CD,186.121834666874,186.941781389892,197.402402426459,225.440493639993,208.999748076,226.876512633529,...,1329.80728479234,1520.21223076132,1538.90167917284,1493.81793829992,1344.10320999294,1522.3933455926,1628.31894446307,1568.15989105305,1673.84113878039,Unnamed: 69
0,Afghanistan,AFG,GDP per capita (current US$),NY.GDP.PCAP.CD,NaN,NaN,NaN,NaN,NaN,NaN,...,522.082216,525.469771,491.337221,496.602504,510.787063,356.496214,357.261153,413.757895,NaN,NaN
1,Africa Western and Central,AFW,GDP per capita (current US$),NY.GDP.PCAP.CD,121.939925,127.454189,133.827044,139.008291,148.549379,155.565216,...,1630.039447,1574.230560,1720.140280,1798.340685,1680.039332,1765.954788,1796.668633,1599.392983,1284.154441,NaN
2,Angola,AGO,GDP per capita (current US$),NY.GDP.PCAP.CD,NaN,NaN,NaN,NaN,NaN,NaN,...,1807.952941,2437.259712,2538.591391,2189.855714,1449.922867,1925.874661,2929.694455,2309.534130,2122.083690,NaN


In [56]:
gdp.tail(3)

,Africa Eastern and Southern,AFE,GDP per capita (current US$),NY.GDP.PCAP.CD,186.121834666874,186.941781389892,197.402402426459,225.440493639993,208.999748076,226.876512633529,...,1329.80728479234,1520.21223076132,1538.90167917284,1493.81793829992,1344.10320999294,1522.3933455926,1628.31894446307,1568.15989105305,1673.84113878039,Unnamed: 69
261,South Africa,ZAF,GDP per capita (current US$),NY.GDP.PCAP.CD,532.147504,545.657512,563.423009,604.536855,645.873376,684.621228,...,5651.205852,6618.335083,6914.178032,6533.711210,5580.603831,6843.399419,6523.410978,6022.542542,6253.371582,NaN
262,Zambia,ZMB,GDP per capita (current US$),NY.GDP.PCAP.CD,221.559849,209.693206,202.281031,203.219451,229.979246,287.425476,...,1239.085279,1483.465773,1463.899979,1258.986198,951.644317,1127.160779,1447.123101,1330.727806,1235.084665,NaN
263,Zimbabwe,ZWE,GDP per capita (current US$),NY.GDP.PCAP.CD,276.419784,279.016489,275.545608,277.005701,281.744539,294.145359,...,1407.420964,3448.086991,2271.852504,1683.913136,1730.453910,1724.387271,2040.546587,2156.034093,2656.409377,NaN


In [57]:
gdp.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 264 entries, 0 to 263
Data columns (total 70 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   Africa Eastern and Southern   264 non-null    object 
 1   AFE                           264 non-null    object 
 2   GDP per capita (current US$)  264 non-null    object 
 3   NY.GDP.PCAP.CD                264 non-null    object 
 4   186.121834666874              150 non-null    float64
 5   186.941781389892              153 non-null    float64
 6   197.402402426459              155 non-null    float64
 7   225.440493639993              155 non-null    float64
 8   208.999748076                 155 non-null    float64
 9   226.876512633529              161 non-null    float64
 10  240.955232436321              162 non-null    float64
 11  243.817323446096              166 non-null    float64
 12  257.190081597614              167 non-null    float64
 13  281.6

## Urban Dataset

## Education Dataset

## Enviroment Dataset

## Meat production Dataset